In [34]:
from pathlib import Path
import requests
import json

In [15]:
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json,text/plain,*/*",
    "Referer": "https://paragens.amp.pt/",
}

In [16]:
session = requests.Session()
session.get("https://paragens.amp.pt/", timeout=10, headers=REQUEST_HEADERS)
session.get("https://paragens.amp.pt/unirmap/", timeout=10, headers=REQUEST_HEADERS)

<Response [200]>

In [27]:
IDOPS = ['aro', 'esp', 'gon', 'mai', 'mat', 'oaz', 'prd', 'prt', 'pov', 'smf', 'str', 'sjm', 'trf', 'vcm', 'vlg', 'vcd', 'vng']
gids = set()
for id_op in IDOPS:
    url = f"https://paragens.amp.pt/unirmap/getmun?idop={id_op}"
    response = session.get(url, timeout=10, headers=REQUEST_HEADERS)
    response.raise_for_status()
    data = response.json()
    for item in data:
        gids.add(item["gid_ida"])
        gids.add(item["gid_volta"])

In [ ]:
Path("geojsons_cache").mkdir(exist_ok=True)
for gid in gids:
    url = f"https://paragens.amp.pt/unirmap/getlinhas?idcarr={gid}"
    response = session.get(url, timeout=10, headers=REQUEST_HEADERS)
    response.raise_for_status()
    data = response.json()
    geodata = { "source": url,
                "geo_line_id": gid,
                "type": "FeatureCollection",
                "features": json.loads(data)
                }
    with open(Path("geojsons_cache") / f"{gid}.geojson", "w") as f:
        json.dump(geodata, f, indent=4)